<a href="https://colab.research.google.com/github/gohard-lab/gohard_ai_upscaler/blob/main/gohard_AI_upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. 🛠️ AI 화질 개선 엔진 설치 (Gohard_AI_Upscaler v2.1)
import os, shutil
from IPython.display import clear_output

!pip install -q streamlit streamlit_javascript uv
!uv pip install basicsr facexlib gfpgan supabase Image pathlib

if os.path.exists('/content/Real-ESRGAN'):
    shutil.rmtree('/content/Real-ESRGAN')

!git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN

%cd /content/Real-ESRGAN

!wget -q -O tracker_web_colab.py https://raw.githubusercontent.com/gohard-lab/ai_image_upscaler/refs/heads/main/src/tracker_web_colab.py

# 4. 의존성 설치 및 패치
!uv pip install -r requirements.txt
!python setup.py develop
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py

clear_output()
print("✅ 폴더 구조 정리 및 설치 완료!")

✅ 폴더 구조 정리 및 설치 완료!


In [ ]:
# @title 2. ⚙️ 화질 개선 및 저장 경로 설정
# @markdown ### 📂 저장 및 워크플로우 설정
Use_Google_Drive = True # @param {type:"boolean"}
# @markdown > 체크 시 결과물과 CSV 리포트가 내 구글 드라이브 `Gohard_Upscaler_Results` 폴더에 영구 저장됩니다.

Mode = "Pro Mode (Batch + Mirroring)" # @param ["Basic Mode (Single)", "Pro Mode (Batch + Mirroring)"]
Target_Canvas = "Instagram (1080x1080)" # @param ["None", "4K (3840x2160)", "Instagram (1080x1080)"]
Background_Color = "Black" # @param ["Black", "White", "Gray"]
Sort_Failures = True # @param {type:"boolean"}
color_map = {"Black": (0,0,0), "White": (255,255,255), "Gray": (128,128,128)}
canvas_map = {"None": None, "4K (3840x2160)": (3840, 2160), "Instagram (1080x1080)": (1080, 1080)}

print(f"✅ 설정 완료: 드라이브 사용({Use_Google_Drive}) / 모드({Mode})")

✅ 설정 완료: 드라이브 사용(True) / 모드(Pro Mode (Batch + Mirroring))


In [ ]:
# @title 3. 🚀 화질 개선 실행 (Gohard_AI_Upscaler v2.2)
import os, shutil, time, csv, cv2, torch
import json
from google.colab import files, drive
from tracker_web_colab import log_app_usage
from pathlib import Path
from PIL import Image
from datetime import datetime

%cd /content/Real-ESRGAN
def start_upscale():
    log_app_usage("upscaler_web", "process_started", details={"canvas": Target_Canvas})

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if Use_Google_Drive:
        drive.mount('/content/drive')
        base_output = Path('/content/drive/MyDrive/Gohard_Upscaler_Results') / timestamp
    else:
        base_output = Path('/content/Real-ESRGAN/results') / timestamp

    upload_dir = Path('/content/Real-ESRGAN/upload')
    log_file = base_output / f"report_{timestamp}.csv"

    if upload_dir.exists(): shutil.rmtree(upload_dir)
    upload_dir.mkdir(parents=True, exist_ok=True)
    base_output.mkdir(parents=True, exist_ok=True)

    print(f"📂 이번 작업 결과는 다음 폴더에 저장됩니다: {base_output}")
    print("👇 복원할 사진들을 선택해주세요 (다중 선택 가능)")
    uploaded = files.upload()

    for filename in uploaded.keys():
        shutil.move(filename, upload_dir / filename)

    with open(log_file, 'w', newline='') as f:
        csv.writer(f).writerow(["Filename", "Canvas", "Time(s)", "Status"])

    images = list(upload_dir.rglob('*'))
    valid_exts = ['.jpg', '.jpeg', '.png']

    start_total = time.time()
    for img_path in images:
        if img_path.suffix.lower() not in valid_exts: continue
        img_start = time.time()

        # 결과 메시지를 담을 변수 초기화
        status = "Unknown"
        error_msg = ""

        try:
            rel_path = img_path.relative_to(upload_dir)
            save_path = base_output / rel_path
            save_path.parent.mkdir(parents=True, exist_ok=True)

            # AI 연산
            !python inference_realesrgan.py -n RealESRGAN_x4plus -i "{img_path}" -o "{save_path.parent}" --outscale 4 --face_enhance --tile 400

            # 캔버스 처리
            res_img_path = save_path.parent / f"{img_path.stem}_out{img_path.suffix}"
            if canvas_map[Target_Canvas]:
                with Image.open(res_img_path) as up_img:
                    canvas = Image.new("RGB", canvas_map[Target_Canvas], color_map[Background_Color])
                    up_img.thumbnail(canvas_map[Target_Canvas], Image.LANCZOS)
                    offset = ((canvas_map[Target_Canvas][0]-up_img.width)//2, (canvas_map[Target_Canvas][1]-up_img.height)//2)
                    canvas.paste(up_img, offset)
                    canvas.save(res_img_path)

            status = "Success"
            error_msg = "None"
        except Exception as e:
            status = "Failed"
            error_msg = str(e)
            if Sort_Failures:
                (base_output / "flagged_failures").mkdir(exist_ok=True)
                shutil.copy(img_path, base_output / "flagged_failures" / img_path.name)

        # 🎯 [핵심] 개별 파일 변환 결과 트래킹 (JSON 컬럼 대응)
        # details에 파일 정보와 성공/실패 여부, 에러 내용을 딕셔너리로 담아 보냅니다. [cite: 2026-03-20]
        log_app_usage(
            app_name="upscaler_web",
            action="file_converted",
            details={
                "filename": img_path.name,
                "status": status,
                "error": error_msg,
                "time_taken": f"{round(time.time() - img_start, 2)}s"
            }
        )

        # CSV 리포트 기록
        elapsed = round(time.time() - img_start, 2)
        with open(log_file, 'a', newline='') as f:
            csv.writer(f).writerow([img_path.name, Target_Canvas, f"{elapsed}s", status])

    print(f"\n✅ 작업 완료! 총 소요시간: {round(time.time()-start_total, 2)}초")
    print(f"📍 최종 저장 위치: {base_output}")

    if not Use_Google_Drive:
        zip_name = f"/content/Gohard_Results_{timestamp}"
        shutil.make_archive(zip_name, 'zip', base_output)
        files.download(f"{zip_name}.zip")

start_upscale()

2026-04-07 13:37:38.058 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:37:38.066 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:37:38.067 Session state does not function when running a script without `streamlit run`
2026-04-07 13:37:38.117 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:37:38.118 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:37:38.119 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:37:38.169 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-07 13:37:38.170 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07

/content/Real-ESRGAN
Mounted at /content/drive
📂 이번 작업 결과는 다음 폴더에 저장됩니다: /content/drive/MyDrive/Gohard_Upscaler_Results/20260407_133738
👇 복원할 사진들을 선택해주세요 (다중 선택 가능)


Saving ligo1447595130.jpg to ligo1447595130.jpg
Saving ligo1449291811.jpg to ligo1449291811.jpg
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" to /content/Real-ESRGAN/weights/RealESRGAN_x4plus.pth

100% 63.9M/63.9M [00:00<00:00, 208MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth" to /content/Real-ESRGAN/gfpgan/weights/detection_Resnet50_Final.pth

100% 104M/104M [00:00<00:

In [ ]:
# 현재 코랩 환경이 '스포츠 모드(GPU)'인지 확인
!nvidia-smi